In [1]:
# Cell 1: Imports and environment setup
from pathlib import Path
import re
import sys

import gymnasium as gym
import imageio.v2 as imageio
import torch
from IPython.display import Video
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.monitor import Monitor

# Resolve project root by searching upward for the folder that contains `vlm`.
start_dir = Path.cwd().resolve()
project_root = next((p for p in [start_dir, *start_dir.parents] if (p / "vlm").is_dir()), start_dir)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from vlm.llava_client import query_llava_position

print(f"Project root: {project_root}")
print(f"Python executable: {sys.executable}")
print(f"CUDA available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "CUDA is required for this notebook (expected RTX 3070)."
device = "cuda"
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Project root: D:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project
Python executable: d:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\final_project_env\Scripts\python.exe
CUDA available: True
Using device: cuda
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
# Cell 2: Reward-shaping logic and callbacks
PROMPT = "Observe the car. Is it on the LEFT slope, RIGHT slope, or BOTTOM? Reply with only one word."

def normalize_label(raw_text: str) -> str:
    # Keep letters only, then normalize to uppercase for robust matching.
    cleaned = re.sub(r"[^A-Za-z]", "", raw_text).upper()
    if "LEFT" in cleaned:
        return "LEFT"
    if "RIGHT" in cleaned:
        return "RIGHT"
    return "BOTTOM"

def label_to_phi(label: str) -> float:
    # Potential reward mapping from VLM label.
    if label == "LEFT":
        return 0.5
    if label == "RIGHT":
        return 1.0
    return 0.0

class VLMRewardShapingWrapper(gym.Wrapper):
    def __init__(self, env, sample_every_n: int = 128, gamma: float = 0.99):
        super().__init__(env)
        self.sample_every_n = sample_every_n
        self.gamma = gamma
        self.step_count = 0
        self.total_steps = 0
        self.current_label = "BOTTOM"
        self.current_phi = 0.0

    def step(self, action):
        obs, reward_env, terminated, truncated, info = self.env.step(action)
        self.step_count += 1
        self.total_steps += 1

        # Dense physical reward: encourage climbing on the right slope.
        position = float(obs[0])
        progress_reward = 0.0
        if position > -0.4:
            progress_reward = (position + 0.4) * 2.0

        # Refresh VLM potential every N steps and decay it over long training.
        if self.step_count >= self.sample_every_n:
            self.step_count = 0
            try:
                frame = self.env.render()
                llava_reply = query_llava_position(frame=frame, model="llava:7b", prompt=PROMPT)
                self.current_label = normalize_label(llava_reply)
                phi = label_to_phi(self.current_label)
                decay = max(0.0, 1.0 - (self.total_steps / 200_000.0))
                self.current_phi = phi * decay
                print(
                    f"[VLM] step={self.total_steps}, raw='{llava_reply}', label={self.current_label}, "
                    f"phi={self.current_phi:.2f}, decay={decay:.3f}"
                )
            except Exception as e:
                self.current_label = "BOTTOM"
                self.current_phi = 0.0
                print(f"[VLM] warning: {e}")

        # Hybrid reward = env reward + decayed VLM potential + right-slope progress.
        reward_total = reward_env + (self.gamma * self.current_phi) + progress_reward
        info["reward_env"] = reward_env
        info["phi"] = self.current_phi
        info["vlm_label"] = self.current_label
        info["progress_reward"] = progress_reward
        info["reward_total"] = reward_total
        return obs, reward_total, terminated, truncated, info

class StopOnFirstSuccessCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)

    def _on_step(self) -> bool:
        # Stop as soon as a high-quality episode reward is better than -160.
        for info in self.locals.get("infos", []):
            if "episode" in info:
                reward = info["episode"]["r"]
                if reward > -160:
                    print(f"High-quality success reached. Episode reward: {reward}. Stopping training.")
                    return False
        return True

class ReachRewardThresholdCallback(BaseCallback):
    def __init__(self, target_reward: float = -160.0, verbose=0):
        super().__init__(verbose)
        self.target_reward = target_reward
        self.first_reach_timestep = None

    def _on_step(self) -> bool:
        # Record the first timestep when target reward is reached.
        if self.first_reach_timestep is not None:
            return True
        for info in self.locals.get("infos", []):
            if "episode" in info:
                episode_reward = info["episode"]["r"]
                if episode_reward >= self.target_reward:
                    self.first_reach_timestep = self.num_timesteps
                    print(
                        f"Reached target reward {self.target_reward} at timestep {self.first_reach_timestep}."
                    )
                    break
        return True

In [ ]:
# Cell 3A: Train, track first success, and save the model
from pathlib import Path
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
import torch

# Callback to track the first time target reward is reached without stopping
class TrackFirstSuccessCallback(BaseCallback):
    def __init__(self, target_reward=-160.0, verbose=0):
        super().__init__(verbose)
        self.target_reward = target_reward
        self.first_reach_timestep = None

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                reward = info["episode"]["r"]
                # Record timestep on first success
                if reward >= self.target_reward and self.first_reach_timestep is None:
                    print(f"\n🎉 Breakthrough! Target {self.target_reward} reached at step: {self.num_timesteps}")
                    self.first_reach_timestep = self.num_timesteps
        return True # Return True to keep training for policy consolidation

log_dir = Path("logs_and_results")
log_dir.mkdir(parents=True, exist_ok=True)

sample_every_n = 128
gamma = 0.99
total_timesteps_cap = 150_000 
baseline_final_steps = 240_000

base_env = gym.make("MountainCar-v0", render_mode="rgb_array")
monitored_env = Monitor(base_env)
train_env = VLMRewardShapingWrapper(monitored_env, sample_every_n=sample_every_n, gamma=gamma)

# Initialize PPO with anti-idling parameters
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    ent_coef=0.05,        # Force exploration
    learning_rate=0.001,  # Higher sensitivity to new rewards
    n_steps=2048,
    verbose=1,
    tensorboard_log=str(log_dir / "tensorboard_vlm_first_success"),
    device="cuda" if torch.cuda.is_available() else "cpu",
)

tracking_callback = TrackFirstSuccessCallback(target_reward=-160.0)

print("🚀 Starting VLM-guided PPO training (Consolidating policy)...")
model.learn(
    total_timesteps=total_timesteps_cap,
    callback=tracking_callback,
    tb_log_name="ppo_mountain_car_vlm_first_success",
)

# === SAVE THE MODEL ===
model_path = log_dir / "mountain_car_vlm_first_success"
model.save(str(model_path))
train_env.close()
print("✅ Training complete. Model saved to: logs_and_results/mountain_car_vlm_first_success.zip")

# === STATISTICS COMPARISON ===
print("\n" + "="*40)
print(f"📊 Baseline steps required: {baseline_final_steps}")

if tracking_callback.first_reach_timestep is None:
    print("⚠️ Training finished, but target -160 was never reached.")
else:
    ours_steps_to_minus_160 = tracking_callback.first_reach_timestep
    speedup = baseline_final_steps / ours_steps_to_minus_160
    print(f"⚡ VLM-PPO first breakthrough step: {ours_steps_to_minus_160}")
    print(f"🚀 Speedup ratio vs Baseline: {speedup:.2f}x")
print("="*40 + "\n")

Using cuda device
Wrapping the env in a DummyVecEnv.
🚀 开始训练 VLM 引导版 PPO，目标：记录首次突破，并巩固成完美策略！
Logging to logs_and_results\tensorboard_vlm_first_success\ppo_mountain_car_vlm_first_success_5


d:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\final_project_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


[VLM] step=128, raw='Right', label=RIGHT, phi=1.00, decay=0.999
[VLM] step=256, raw='Left', label=LEFT, phi=0.50, decay=0.999
[VLM] step=384, raw='Left', label=LEFT, phi=0.50, decay=0.998
[VLM] step=512, raw='Right', label=RIGHT, phi=1.00, decay=0.997
[VLM] step=640, raw='Left', label=LEFT, phi=0.50, decay=0.997
[VLM] step=768, raw='Right', label=RIGHT, phi=1.00, decay=0.996
[VLM] step=896, raw='Left', label=LEFT, phi=0.50, decay=0.996
[VLM] step=1024, raw='Left', label=LEFT, phi=0.50, decay=0.995
[VLM] step=1152, raw='Left', label=LEFT, phi=0.50, decay=0.994
[VLM] step=1280, raw='Left', label=LEFT, phi=0.50, decay=0.994
[VLM] step=1408, raw='Right', label=RIGHT, phi=0.99, decay=0.993
[VLM] step=1536, raw='Left', label=LEFT, phi=0.50, decay=0.992
[VLM] step=1664, raw='Left', label=LEFT, phi=0.50, decay=0.992
[VLM] step=1792, raw='Right', label=RIGHT, phi=0.99, decay=0.991
[VLM] step=1920, raw='Left', label=LEFT, phi=0.50, decay=0.990
[VLM] step=2048, raw='Left', label=LEFT, phi=0.49, d

d:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\final_project_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


✅ 完美通关视频生成完毕！


In [ ]:
# Cell 3B: Load the saved model and export a demo video
from pathlib import Path
import gymnasium as gym
from stable_baselines3 import PPO
import imageio.v2 as imageio
from IPython.display import Video
import torch

log_dir = Path("logs_and_results")
model_path = log_dir / "mountain_car_vlm_first_success.zip"

print("🎥 Generating final demo video using the consolidated model...")
video_env = gym.make("MountainCar-v0", render_mode="rgb_array")

# Load the saved model from disk
loaded_model = PPO.load(str(model_path), device="cuda" if torch.cuda.is_available() else "cpu")

frames = []
obs, info = video_env.reset()
terminated = False
truncated = False

# Run one full episode
while not (terminated or truncated):
    frames.append(video_env.render())
    # deterministic=True ensures it uses the optimal policy without random exploration
    action, _ = loaded_model.predict(obs, deterministic=True) 
    obs, reward, terminated, truncated, info = video_env.step(action)

video_env.close()

# Save frames to MP4
video_path = log_dir / "mountain_car_vlm_first_success_demo.mp4"
imageio.mimsave(video_path, frames, fps=30)

print("✅ Video generated successfully!")
display(Video(str(video_path), embed=True))